# Training models in half genotypes to infer on others

## Loading the Database

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 8
NUM_CLASSES = 1
CROP = "sorghum"  # "wheat" or "sorghum"
GENOTYPES = 

dataset_path = f"../data/{CROP}/"

class SegmentationDataset(Dataset):
    def __init__(self, root_dir, train=True, collection_dates=None):
        self.root_dir = root_dir
        self.train = train
        self.img_dir = os.path.join(root_dir, "train/images" if train else "test/images")
        self.mask_dir = os.path.join(root_dir, "train/masks" if train else "test/masks")

        # List all image files
        image_files = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        mask_files = sorted([
            f for f in os.listdir(self.mask_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        
        # Filter by collection date if provided
        if collection_dates is not None:
            # Accept single date or list of dates
            if isinstance(collection_dates, (str, int)):
                collection_dates = [str(collection_dates)]
            image_files = [f for f in image_files if f.split('_')[0] in collection_dates]
            mask_files = [f for f in mask_files if f.split('_')[0] in collection_dates]
        print(image_files)
        # Ensure matching images/masks
        assert len(image_files) == len(mask_files), \
            f"Image/Mask count mismatch: {len(image_files)} vs {len(mask_files)}"

        self.image_files = image_files
        self.mask_files = mask_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = torch.from_numpy(np.array(mask)).float().unsqueeze(0)
        mask = (mask > 0).float()

        return image, mask

train_dataset = SegmentationDataset(dataset_path, train=True, collection_dates=COLLECTION_DATE)
test_dataset = SegmentationDataset(dataset_path, train=False, collection_dates=COLLECTION_DATE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")